# EDA — ISO New England Hourly Demand

Exploration workbench for the raw pull in `data/raw/iso_ne_demand_2025-08-12_2026-08-12.csv`. This is scratch space: explore distributions, gaps, outliers, and seasonality here, and prototype cleaning/transformation approaches before codifying anything into `src/features/`.

The raw CSV stays untouched — nothing in this notebook should overwrite `data/raw/`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/iso_ne_demand_2025-08-12_2026-08-12.csv", parse_dates=["period"])
df.shape

(8784, 7)

In [2]:
df.head()

,period,respondent,respondent-name,type,type-name,value,value-units
0,2025-08-12 00:00:00,ISNE,ISO New England,D,Demand,22159,megawatthours
1,2025-08-12 01:00:00,ISNE,ISO New England,D,Demand,21313,megawatthours
2,2025-08-12 02:00:00,ISNE,ISO New England,D,Demand,19888,megawatthours
3,2025-08-12 03:00:00,ISNE,ISO New England,D,Demand,18365,megawatthours
4,2025-08-12 04:00:00,ISNE,ISO New England,D,Demand,16988,megawatthours


In [3]:
from dotenv import load_dotenv
load_dotenv("/Users/shubhantamhane/energy-demand-forecast/.env")
from auto_insights import InsightsGenerator

In [4]:
report = InsightsGenerator(df, dataset_name='Energy Demand', run_llm=True).run()
print("Done in", report.elapsed_seconds, "seconds")
print("Figures generated:", report.list_figures())
print(report.stats["overview"])

[INFO] auto_insights.core: ============================================================
[INFO] auto_insights.core: InsightsGenerator: starting pipeline for 'Energy Demand'
[INFO] auto_insights.core:   shape   : 8784 rows × 7 cols
[INFO] auto_insights.core:   run_llm : True  |  run_viz: True  |  run_report: True
[INFO] auto_insights.core: ============================================================
[INFO] auto_insights.core: ── step: stats
[INFO] auto_insights.stats: Profiling DataFrame: 8784 rows × 7 cols  (numeric=1, categorical=5, datetime=1, boolean=0, text=0)
[INFO] auto_insights.stats: Profiling complete.
[INFO] auto_insights.core:    stats done  (0.63s)
[INFO] auto_insights.core: ── step: viz
[INFO] auto_insights.viz: viz: generated 8 figures.
[INFO] auto_insights.core:    viz done  (0.50s)
[INFO] auto_insights.core: ── step: correlations
[WARNING] auto_insights.corr: corr: fewer than 2 numeric columns — skipping correlation analysis.
[INFO] auto_insights.core:    correlations do

Done in 27.39 seconds
Figures generated: ['bar_respondent', 'bar_respondent-name', 'bar_type', 'bar_type-name', 'bar_value-units', 'box_value', 'hist_value', 'numeric_overview']
{'n_rows': 8784, 'n_cols': 7, 'total_cells': 61488, 'total_nulls': 0, 'null_pct_overall': 0.0, 'duplicate_rows': 0, 'column_type_counts': {'numeric': 1, 'categorical': 5, 'datetime': 1}, 'columns_with_nulls': {}}


## Record type check

The pull was filtered to `type=D` (Demand) at the API level, so this checks whether anything besides Demand slipped in — one bar for Demand, one for everything else.

In [ ]:
import matplotlib.pyplot as plt

counts = df["type-name"].value_counts()
demand_count = counts.get("Demand", 0)
other_count = counts.drop("Demand", errors="ignore").sum()

labels = ["Demand", "Other"]
values = [demand_count, other_count]
colors = ["#2a78d6", "#eb6834"]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values, color=colors, width=0.5)

for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{value:,}", ha="center", va="bottom", fontsize=10)

ax.set_ylabel("Row count")
ax.set_title("Record type: Demand vs. Other")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Staging note: drop constant columns

`respondent`, `respondent-name`, `type`, `type-name`, and `value-units` are constant across every row in this pull (confirmed above) — they're just the API query echoed back, not real information. Only `period` and `value` vary.

**Decision:** drop these constant columns when staging the data for the final step (i.e. in the `data/processed` output / `src/features/` pipeline), not in `data/raw` — the raw file stays untouched as the source of truth. Demonstrated below for reference.

In [ ]:
constant_cols = ["respondent", "respondent-name", "type", "type-name", "value-units"]
df_staged_preview = df.drop(columns=constant_cols)
df_staged_preview.head()

## Demand over the year, with solstices highlighted

Line graph of `value` across the full pull (2025-08-12 to 2026-08-12), months on the y-axis and demand on the x-axis. Winter solstice (2025-12-21) and summer solstice (2026-06-21) are marked with reference lines.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

daily = df.set_index("period")["value"].resample("D").mean().reset_index()

fig, ax = plt.subplots(figsize=(7, 9))

ax.plot(daily["value"], daily["period"], color="#2a78d6", linewidth=1.5)

solstices = [
    (pd.Timestamp("2025-12-21"), "Winter solstice (Dec 21)"),
    (pd.Timestamp("2026-06-21"), "Summer solstice (Jun 21)"),
]
for date, label in solstices:
    ax.axhline(date, color="#eb6834", linewidth=1.5, linestyle="--")
    ax.text(daily["value"].max(), date, f"  {label}",
            color="#eb6834", va="center", ha="left", fontsize=9)

ax.yaxis.set_major_locator(mdates.MonthLocator())
ax.yaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

ax.set_xlabel("Daily average demand (MWh)")
ax.set_ylabel("Month")
ax.set_title("ISO-NE Daily Average Demand, Aug 2025\u2013Aug 2026")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()